# Proyek UAS PMDPM

<b>Harap jangan menghapus cell dan komentar yang diberikan!</b>

Setiap cell markdown dan code akan berisi instruksi pengerjaan Proyek UAS PMDPM

Dalam notebook ini Anda akan diminta untuk membuat sebuah proyek Pembelajaran Mesin menggunakan dataset yang sudah disediakan. Proyek akan terdiri dari proses inisialisasi, data loading, data cleansing, feature engineering, modelling dan evaluasi model. Pada bagian akhir silahkan berikan laporan singkat dari proyek yang sudah dikerjakan

## Penjelasan Tugas

Terapkan pemrosesan dataset dari loading, data cleansing (jika perlu), data scaling, feature selection, hingga evaluasi model secara sederhana menggunakan code yang sudah dipelajari dalam latihan-latihan dan tugas.

### Inisialisasi

Bagian berikut berisi import library yang dibutuhkan

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV,StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report,roc_auc_score, roc_curve
from sklearn.multiclass import OneVsRestClassifier

### Data Loading

Bagian berikut berisi proses data loading (boleh dengan file upload atau dengan mount drive), jangan lupa tampilkan 20 baris data teratas!

In [ ]:
#loading data yang akan diolah
data = pd.read_csv('paris_housing2000.csv')
data.head(20)

In [ ]:
#bersihkan dataset menggunakan metode yang sudah dipelajari
# detect row yang mengandung NaN
data.isnull().sum()
# tidak ada data yang null, tidak perlu untuk menghapus data

### Data Visualization

Dalam proses visualisasi data, silahkan visualisasikan feature menggunakan plot yang sesuai agar mendapat insight dari data dengan menjawab pertanyaan-pertanyaan berikut:

    untuk sub A:
    1. Bagaimana perbandingan median luas basement rumah berdasarkan tipe rumahnya (Basic dan Luxury)?
    2. Bagaimana komposisi jumlah rumah yang baru dibangun atau tidak pada tipe rumah Luxury?
   

    untuk sub B:
    1. Bagaimana perbandingan median luas loteng rumah berdasarkan tipe rumahnya (Basic dan Luxury)
    2. Bagaimana komposisi jumlah rumah yang memiliki pelindung badai atau tidak pada tipe rumah Basic?
   

In [ ]:
#visualisasi fitur menggunakan metode yang sudah dipelajari
# pebandingan median luas basement berdasarkan apakah rumahnya basic atau luxury
sns.boxplot(data=data, x="category", y="basement")

luas_basement_basic = data[data['category'] == 'Basic']['basement'].median()
luas_basement_luxury = data[data['category'] == 'Luxury']['basement'].median()

print("Median luas basement rumah basic: ", luas_basement_basic)
print("Median luas basement rumah luxury: ", luas_basement_luxury)

In [ ]:
# komposisi jumlah rumah yang baru dibangun atau tidak pada rumah luxury
luxury_new = data[data['category'] == 'Luxury'][data['isNewBuilt'] == 'Y'].shape[0]
luxury_old = data[data['category'] == 'Luxury'][data['isNewBuilt'] == 'N'].shape[0]
print(luxury_old)

komposisi = [luxury_new, luxury_old]
label = ['Newly built', 'Not newly built']

plt.pie(komposisi, labels=label, autopct='%1.1f%%')

### Deskripsi Data Visualization
Berikan keterangan atau deskripsi singkat mengenai informasi yang diperoleh dari visualisasi data tersebut. Jawablah pertanyaan di atas pada cell markdown ini!

1. Perbandingan median luas basement rumahnya adalah basic memiliki luas basement yang lebih tinggi dibanding luxury. Dapat dilihat dalam box plot dan output median bahwa basic memiliki median 5170, sedangkan luxury hanya mempunyai median sebanyak 4847.5 .

2. Komposisi rumah luxury yang baru dibangun atau tidak, jika dilihat dalam plot maka dapat disimpulkan bahwa semua rumah bertipe luxury adalah rumah yang barusan dibangun (newly built).

### Data Cleansing

Bagian berikut berisi proses pembersihan data (jika diperlukan, minimal cek apakah ada data yang kosong atau duplikat). Drop feature yang berlaku sebagai unique identifier (CityCode) dan drop feature Price. Lakukan OneHotEncoder untuk data kategorik string. Pisahkan juga label dengan feature!

In [ ]:
# detect row yang mengandung NaN
print(data.isnull().sum())
# tidak ada data yang null, tidak perlu untuk menghapus data

# drop kolom price
data = data.drop(['price'], axis=1)

In [ ]:
# encode data dengan one hot encoding
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer

cat_cols = ['isNewBuilt', 'hasStormProtector', 'hasStorageRoom', 'category']

transformer = make_column_transformer((OneHotEncoder(), cat_cols), remainder='passthrough')

data_enc = transformer.fit_transform(data)

print(data_enc.shape)
print(transformer.get_feature_names_out())

data_enc = pd.DataFrame(data_enc, columns=transformer.get_feature_names_out())

### Train-test split

> Untuk sub A: random_state train_test_split gunakan 2 digit terakhir NPM terbesar di sub kelompok  

> Untuk sub B: random_state train_test_split gunakan 2 digit terakhir NPM terkecil di sub kelompok 

In [ ]:
#train-test split dan data scaling
from sklearn.model_selection import train_test_split

X = data_enc.drop(['onehotencoder__category_Basic', 'onehotencoder__category_Luxury'], axis=1)
y = data['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=11) # npm 200710880 dan 210711011, diambil terbesar -> 210711011, diambil 2 digit terahir -> 11.

### Modelling

Gunakan Pipeline untuk modelling dan grid search cross-validation untuk pemilihan model terbaik. Gunakan Stratified K-fold Cross Validation dengan nilai n_splits=5. Pipeline terdiri atas data scaling, feature selection, dan classifier. Buatlah 2 Pipeline dan 2 GridSearchCV untuk masing-masing kombinasi berikut.

>Step data scaling untuk membandingkan perlunya data scaling atau tidak. Gunakan dua nilai pada parameter grid yaitu: [None,Scaler]. Untuk sub A: gunakan StandardScaler. Untuk sub B: gunakan MinMaxScaler

>Step feature selection untuk membandingkan metode feature selection yang digunakan. Gunakan dua metode pada parameter grid yaitu: [Selector]. Untuk sub A: gunakan SelectKBest dengan ketentuan k minimal 2, maksimal n-1. Untuk sub B: gunakan SelectPercentile dengan ketentuan persentil minimal 30%, maksimal 80%. Kalian juga perlu melakukan setting parameter grid nilai k atau percentile-nya.

>Step classifier untuk membandingkan algoritme classifier. Eksplorasi pengaturan parameter yang ada pada algoritme yang akan Anda gunakan, tidak terbatas pada 1-2 parameter saja supaya optimal (selain random_state). Silahkan bereksperimen untuk parameter seperti n_estimators, max_depth untuk tree-based classifier; C untuk Logistic Regression; dan C, gamma, serta kernel untuk SVM agar mendapat akurasi terbaik.

> Sub A: RandomForestClassifier(n_estimators=bebas minimal 50, random_state= dua digit terakhir NPM terbesar dalam sub kelompok, max_depth=bebas, criterion='entropy') dan LogisticRegression(C=bebas, max_iter=10000, class_weight='balanced')

> Sub B: K-Nearest Neighbors(n_neighbors=bebas, weights='distance') dan SVC(C=bebas, gamma=bebas, class_weight='balanced')

>Tampilkan best estimator dan nilai best score dari kedua grid-search cross-validation. Fit best estimator pada train set kemudian ujikan pada test set dengan fungsi score. Format score 3 digit di belakang koma (.3f).

In [ ]:
#cell code ini untuk proses pembuatan model pipeline & grid search 1 (scaler none dan standardscaler, feature selection skbest, classifier random forest)
pipe_rndForest= Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest()),
    ('classifier', RandomForestClassifier(random_state=11, criterion='entropy'))
])
pipe_rndForest.fit(X_train, y_train)

param_grid_rndForest = {
    'scaler': [None, StandardScaler()],
    'feature_selection__k': np.arange(2, X_train.shape[1]-1),
    'classifier__n_estimators': [50, 150, 200],
    'classifier__max_depth': [10, 50, 100],
}

skfold = StratifiedKFold(n_splits=5)
GSCV_rndForest = GridSearchCV(pipe_rndForest, param_grid_rndForest, cv=skfold)
GSCV_rndForest.fit(X_train, y_train)

print("Best params: {}".format(GSCV_rndForest.best_params_))
print("Best CV score: {:.3f}".format(GSCV_rndForest.best_score_))
print("Test set score: {:.3f}".format(GSCV_rndForest.score(X_test,y_test)))
mask = GSCV_rndForest.best_estimator_.named_steps['feature_selection'].get_support()
selected_feat = X.columns[mask]
print("Selected features:",selected_feat)

In [ ]:
#cell code ini untuk proses pembuatan model pipeline & grid search 2 (scaler none dan standardscaler, feature selection skbest, classifier logistic regression)
pipe_logReg= Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest()),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
pipe_logReg.fit(X_train, y_train)

param_grid_logReg = {
    'scaler': [None, StandardScaler()],
    'feature_selection__k': np.arange(2, X_train.shape[1]-1),
    'classifier__C': [0.1, 1, 2, 5, 10],
}

skfold = StratifiedKFold(n_splits=5)
GSCV_logReg = GridSearchCV(pipe_logReg, param_grid_logReg, cv=skfold)
GSCV_logReg.fit(X_train, y_train)

print("Best params: {}".format(GSCV_logReg.best_params_))
print("Best CV score: {:.3f}".format(GSCV_logReg.best_score_))
print("Test set score: {:.3f}".format(GSCV_logReg.score(X_test,y_test)))
mask = GSCV_logReg.best_estimator_.named_steps['feature_selection'].get_support()
selected_feat = X.columns[mask]
print("Selected features:",selected_feat)

### Evaluasi Model

Bagian berikut berisi evaluasi model menggunakan classification report dan skor AUC. Tampilkan classification report dan AUC dari best estimator yang diperoleh masing-masing grid search!

In [ ]:
#lakukan proses evaluasi model algoritme pembelajaran mesin
gs_pred_rndForest = GSCV_rndForest.predict(X_test)
gs_pred_logReg = GSCV_logReg.predict(X_test)

print("======== Random Forest Classifier ========")
print("Confusion Matrix:\n", confusion_matrix(y_test, gs_pred_rndForest))
print("Classification Report:\n", classification_report(y_test, gs_pred_rndForest, target_names=['Basic', 'Luxury']))

clf_rndForest = OneVsRestClassifier(GSCV_rndForest.best_estimator_)
clf_rndForest.fit(X_train, y_train)
clf_rndForest_pred = clf_rndForest.predict(X_test)
print("\nAUC score: {:.3f}".format(roc_auc_score(y_test, clf_rndForest.predict_proba(X_test)[:,1], multi_class='ovr')))


print("\n\n======== Logistic Regression Classifier ========")
print("Confusion Matrix:\n", confusion_matrix(y_test, gs_pred_logReg))
print("Classification Report:\n", classification_report(y_test, gs_pred_logReg, target_names=['Basic', 'Luxury']))

clf_logReg = OneVsRestClassifier(GSCV_logReg.best_estimator_)
clf_logReg.fit(X_train, y_train)
clf_rndForest_pred = clf_logReg.predict(X_test)
print("\nAUC score: {:.3f}".format(roc_auc_score(y_test, clf_logReg.predict_proba(X_test)[:,1], multi_class='ovr')))

## Kesimpulan

Silahkan jawab pertanyaan berikut:

1. Apa saja feature-feature yang penting untuk membedakan antara rumah basic dan luxury? 

2. Model apakah yang memiliki performa paling baik terhadap dataset? Apakah parameter setting yang paling optimal dari model tersebut?

*tulis jawaban Anda di cell ini*
1. Feature yang membedakan antara rumah basic dan luxury diantaranya adalah apakah rumah tersebut barusan dibangun (newly built), apakah mempunyai proteksi dari badai, apakah mempunyai ruang penyimpanan, jumlah dari owner sebelumnya, luas basement, dan luas loteng. Fitur yang paling penting terletak di apakah rumah tersebut baru dibangun atau tidak, karena menurut model random forest yang telah dijalankan, fitur yang terpilih adalah kedua fitur tersebut.

2. Kedua model memiliki performa yang baik, tetapi model yang memiliki performa paling baik adalah model logistic regresssion yang memiliki auc score sebesar 0.795 (dibanding dengan auc score random forest sebesar 0.786). Parameter yang paling optimal untuk model tersebut adalah C = 5 untuk classifier logistic regression, k=9 untuk skbest, dan none untuk scaler (tidak memakai/melakukan scaling data).